# **PRÉ-PROCESSAMENTO**

##### Importando bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import plotly.figure_factory as ff
import seaborn as sns
import matplotlib.pyplot as plt
import os

##### Lendo o arquivo tratado

In [ ]:
df = pd.read_csv('../data/csv/processed/heart_tratado.csv', encoding='utf-8')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df_2 = df.copy()

##### Codificação das variáveis categóricas manualmente

In [ ]:
df_2['Sex'] = df_2['Sex'].replace({'M': 0, 'F':1})
df_2['ChestPainType'] = df_2['ChestPainType'].replace({'TA':0, 'ATA':1, 'NAP': 2, 'ASY':3})
df_2['RestingECG'] = df_2['RestingECG'].replace({'Normal':0, 'ST':1, 'LVH':2})
df_2['ExerciseAngina'] = df_2['ExerciseAngina'].replace({'N':0, 'Y':1})
df_2['ST_Slope'] = df_2['ST_Slope'].replace({'Up':0, 'Flat': 1, 'Down':2})

In [ ]:
df_2.head()

In [ ]:
df_2.dtypes

In [ ]:
df_2.shape

**LEGENDA**

* Age = Idade (anos)  
* Sex = Sexo (0=M, 1=F)  
* Chest Pain Type = Tipo de dor no peito (0=TA: Tangina Atípica; 1=ATA: * Angina Atípica; 2=NAP: Dor Não Anginosa; 3=ASY: Assintomático)  
* Resting BP = Pressão sanguínia em repouso (mmHg)  
* Cholesterol = Colesterol sérico (mg/dl)  
* Fasting BS = Açucar no sangue em jejum (mg/dl) (0=Fasting BS < 120 (não diabético); 1=Fasting BS>= 120 (diabético))  
* Resting ECG = Eletrocardiograma em repouso (0=Normal; 1=ST: * Anormalidade de onda ST-T; 2=LVH: Hipertrofia ventricular esquerda)  
* MaxHR = Frequência cardiáca máxima  
* Exercise Angina = Angina indusida por exercício (0=N; 1=Y)  
* Old Peak = Depressão de ST induzida por exercício em relação ao repouso  
* ST_Slope = Inclinação do seguimento ST (0=UP; 1=Flat; 2=Down)  
* Heart Disease = Doença cardíaca (0=Não possui; 1=Possui)  

## **ATRIBUTOS PREVISORES E ALVO**

In [ ]:
previsores = df_2.iloc[:, 0:11].values # iloc -> localiza pelo índice [todas as linhas, 10 primeiras colunas]
previsores

In [ ]:
previsores.shape

In [ ]:
target = df_2.iloc[:, 11] # Target -> alvo
target

In [ ]:
target.shape

### **Análise das escalas (Escalonamento)**

Escalonamento busca fazer com que média fique próxima de zero e o desvio padrão fique próximo de 1 (como uma distribuição normal padrão).

In [ ]:
df_2.describe()

Padronização (utiliza média e desvio padrão como referência)  
Normalização (utiliza os valores máximo e mínimo como referência)

In [ ]:
from sklearn.preprocessing import StandardScaler # Padronização (StandardScaler) -> escala z-score. Média 0 e desvio padrão 1.
# Já a Normalização (MinMaxScaler) deixa os valores entre 0 e 1.

In [ ]:
previsores_esc = StandardScaler().fit_transform(previsores) #Transformação de previsores quanto a sua escala utilizando padronização
previsores_esc

In [ ]:
previsores_df = pd.DataFrame(previsores_esc) # Transforma o previsores_esc em um dataframe
previsores_df

In [ ]:
previsores_df.mean()

In [ ]:
previsores_df.describe()

In [ ]:
df.head()

#### **Codificação de Variáveis Categóricas**

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
previsores2 = df.iloc[:,0:11].values
previsores2


In [ ]:
# A primeira linha sempre com previsores, pois está transformando a matriz original, nas camadas subsequentes, se refere a matriz já transformada
previsores2[:,1] = LabelEncoder().fit_transform(previsores2[:,1])
previsores2[:,2] = LabelEncoder().fit_transform(previsores2[:,2])
previsores2[:,6] = LabelEncoder().fit_transform(previsores2[:,6])
previsores2[:,8] = LabelEncoder().fit_transform(previsores2[:,8])
previsores2[:,10] = LabelEncoder().fit_transform(previsores2[:,10])
previsores2

##### **OneHotEncoder - Criação de variáveis Dummy (fictícia)**

>⚠️**Cuidado** com multicolinearidade (variáveis altamente corelacionadas entre si)

Variáveis dummy (fictícias) são criadas para evitar multicolinearidade. Multicolinearidade ocorre quando duas ou mais variáveis independentes em um modelo de regressão estão altamente correlacionadas, o que pode distorcer os resultados do modelo. Assim é possível eliminar uma das variáveis dummy para cada categoria, reduzindo a redundância e melhorando a interpretabilidade do modelo.

<p><strong>Cuidado com a multicolinearidade</strong> (variáveis altamente correlacionadas entre si).</p>

<table style="border:1px solid #ccc; border-collapse:collapse;">
<tr>
<td style="padding:10px; vertical-align:top;">
<strong>Você faz atividade física?</strong>
</td>
<td style="padding:10px;">
<table style="border-collapse:collapse;">
<tr><td><b>A = 0</b></td><td>Não.</td></tr>
<tr><td><b>B = 1</b></td><td>Sim, um ou dois dias por semana.</td></tr>
<tr><td><b>C = 2</b></td><td>Sim, três ou quatro dias por semana.</td></tr>
<tr><td><b>D = 3</b></td><td>Sim, pelo menos cinco dias por semana.</td></tr>
</table>
</td>
</tr>
</table>

<pre>
A B C D
1 0 0 0
0 1 0 0
0 0 1 0
0 0 0 1
</pre>


In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

**Parâmetros ColumnTransformer**

* name: nome dado a transformação
* transformer: tipo de transformador (OneHotEncoder)
* columns: colunas a serem transformadas
* remaider: o que acontecerá com o restante das colunas não relacionadas: 1) drop = exclui as outras colunas; 2) passthrough = mantém as outras colunas. drop é o default
* sparse_threshold: parâmetro classificador de matrizes esparsas: default = 0.3
* n_jobs: número de trabalhos a serem executados em paralelo: default -> nenhum
* transformer_weights: definição dos pesos aos transformadores
* verbose: default = False; se True a execução é apresentada na tela

In [ ]:
previsores3 = ColumnTransformer(transformers=[('OneHot', OneHotEncoder(), [1,2,6,8,10])], remainder='passthrough', verbose=True).fit_transform(previsores2)

In [ ]:
previsores3

In [ ]:
previsores3.shape

In [ ]:
previsores3_df= pd.DataFrame(previsores3)
previsores3_df.head()

In [ ]:
df.head() # Original DataFrame

##### **Escalonamento de previsores3**

In [ ]:
previsores3_esc = StandardScaler().fit_transform(previsores3)
previsores3_esc

In [ ]:
previsores3_esc_df = pd.DataFrame(previsores3_esc)
previsores3_esc_df

In [ ]:
previsores3_esc_df.describe()

**RESUMO PRÉ-PROCESSAMENTO**

* target = variável que se pretende atingir (tem ou não doença cardiáca);
* previsores = conjunto de variáveis previsoras junto com as variáveis categóricas transformadas em numéricas, sem escalonar;
* previsores_esc = conjunto de variáveis previsoras junto com as variáveis categóricas transformadas em numéricas escalonadas;
* previsores2 = conjunto das variáveis previsoras com as categóricas transformadas em numéricas pelo LabelEncoder;
* previsores3 = conjunto de variáveis previsoras transformadas pelo LabelEncoder e OneHotEncoder, sem escalonar;
* previsores3_esc = conjunto de variáveis previsoras transformadas pelo LabelEncoder e OneHotEncoder escalonadas


### **Salvando Variáveis (Atributos)**

In [ ]:
import pickle
os.makedirs('../pkl/', exist_ok=True)

In [ ]:
# Criando o arquivo para salvar os previsores
arq_target = open('../pkl/heart.pkl', 'wb')

In [ ]:
# Salvando a variável previsores no arquivo
pickle.dump(target, arq_target)

In [ ]:
# Fechando o arquivo!! ISSO É IMPORTANTE
arq_target.close()

In [ ]:
# Lendo o arquivo de previsores
arq_target = open('../pkl/heart.pkl', 'rb')

In [ ]:
target = pickle.load(arq_target)

In [ ]:
np.array(target)

In [ ]:
# Criando o arquivos pkl para salvar as outras variáveis
arq_previsores = open('../pkl/heart_previsores.pkl', 'wb')
# Salvando a variável previsores no arquivo
pickle.dump(previsores, arq_previsores)
arq_previsores_esc = open('../pkl/heart_previsores_esc.pkl', 'wb')
pickle.dump(previsores_esc, arq_previsores_esc)
arq_previsores2 = open('../pkl/heart_previsores2.pkl', 'wb')
pickle.dump(previsores2, arq_previsores2)
arq_previsores3 = open('../pkl/heart_previsores3.pkl', 'wb')
pickle.dump(previsores3, arq_previsores3)
arq_previsores3_esc = open('../pkl/heart_previsores3_esc.pkl', 'wb')
pickle.dump(previsores3_esc, arq_previsores3_esc)
# Fechando o arquivo!! ISSO É IMPORTANTE
arq_previsores.close()
arq_previsores_esc.close()
arq_previsores2.close()
arq_previsores3.close()
arq_previsores3_esc.close()

In [ ]:
# Lendo arq_previsores
arq_previsores = open('../pkl/heart_previsores.pkl', 'rb')
arq_previsores = pickle.load(arq_previsores)
arq_previsores

# **ALGORITMOS DE MACHINE LEARNING**

## **Base de Treino e Teste**

In [ ]:
from sklearn.model_selection import train_test_split

**Parâmetros train_test_split**

* **arrays:** nomes dos atributos previsores e target;
* **test_size:** tamanho em % dos dados de teste. Default = None;
* **train_size:** tamanho em % dos dados de treino. Default = None;
* **random_state:** nomeação de um estado aleatório;
* **shuffle:** embaralhamento dos dados aleatórios. Associado ao random_state ocorre o mesmmo embaralhamento sempre. Default = True;
* **stratify:** possibilidade de dividir os dados de forma estratificada. Default é None (neste caso, é mantida a proporção, isto é, sem tem 30% de zeros e 70% de 1 no dataframe, a separação de treinamento e teste se manterá nessa proporção).

In [ ]:
previsor_usado = previsores3_esc
# x_treino -> atributos de treino, y_treino -> target de treino, x_teste -> atributos de teste, y_teste -> target de teste
x_treino, x_teste, y_treino, y_teste = train_test_split(previsor_usado, target, test_size=0.3, random_state=0)

In [ ]:
x_teste.shape

In [ ]:
# Soma dos valores de treino e teste têm que ser igual ao total de valores do dataset original
print(f'''Atributos: {x_treino.shape[0]} valores para treino e {x_teste.shape[0]} valores para teste
      Target: {y_treino.shape[0]} valores para treino e {y_teste.shape[0]} valores para teste
      ''')

## **Naive Bayes**


[Documentação do Naive Bayes](https://scikit-learn.org/stable/modules/naive_bayes.html)

---
🧠 **Mini-Resumo: Algoritmo Naive Bayes**

O **Naive Bayes** é um algoritmo de **classificação probabilística** baseado no **Teorema de Bayes**, usado para prever a classe de uma amostra com base nas probabilidades condicionais dos atributos.

---

⚙️ **Conceito-Chave**

Ele parte da ideia de que todos os atributos são **independentes entre si** — uma suposição “ingênua” (*naive*), mas que simplifica muito o cálculo e funciona bem em muitos casos.
$$

P(C|X) = \frac{P(X|C) \cdot P(C)}{P(X)}

$$
onde:

* Probabilidade da classe ( C ) dado o vetor de atributos ( X ):
$$
P(C|X)
$$
* Probabilidade de ( X ) ocorrer dentro da classe ( C ):
$$
P(X|C)
$$
* Probabilidade prévia da classe:
$$
P(C)
$$
* Probabilidade total dos atributos:
$$
P(X)
$$

---

🧩 **Variações**

* **Gaussian Naive Bayes** → para atributos contínuos (assume distribuição normal);
* **Multinomial Naive Bayes** → ideal para contagem de palavras, como em *NLP*;
* **Bernoulli Naive Bayes** → usado para variáveis binárias (presença/ausência).

---


💡 **Em resumo:**

> Um classificador simples, rápido e surpreendentemente eficaz, mesmo quando a suposição de independência não é totalmente verdadeira.

---

**APLICAÇÕES:**
* Filtros de span;
* Diagnósticos médicos;
* Classificação de informações textuais;
* Análise de crédito;
* Separação de documentos;
* Previsão de falhas.
---
**Vantagens:**
* Rápido e de fácil entendimento;
* Pouco esforço computacional;
* Bom desempenho com muitos dados;
* Boas previsões com poucos dados.
---
**Desvantagens:**
* Considera atributos independentes;
* Atribui valor nulo de probabilidade quando uma classe contida no conjunto de teste não se apresenta no conjunto de treino.

##### **Treinamento do Algoritmo**

In [ ]:
from sklearn.naive_bayes import GaussianNB

In [ ]:
naive = GaussianNB()
naive.fit(x_treino, y_treino)

#### **Avaliação do Algortimo**

##### **Análise de Teste**

In [ ]:
previsoes_teste_naive = naive.predict(x_teste)
previsoes_teste_naive

In [ ]:
y_teste.values

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# Determina a acurácia do modelo
acuracia_teste = accuracy_score(y_teste, previsoes_teste_naive)
acuracia_teste

In [ ]:
print(f'Acurácia do Naive Bayes (teste) é de: {acuracia_teste*100:.2f}%')

In [ ]:
# Matriz de confusão de teste. Mostra os verdadeiros positivos, verdadeiros negativos, falsos positivos e falsos negativos
matriz_confusao_teste_nb = confusion_matrix(y_teste, previsoes_teste_naive)
print(f'Matriz de Confusão:\n{matriz_confusao_teste_nb}')
print(f'{matriz_confusao_teste_nb[0,0]} true negatives\n{matriz_confusao_teste_nb[0,1]} false positives\n{matriz_confusao_teste_nb[1,0]} false negatives\n{matriz_confusao_teste_nb[1,1]} true positives')

In [ ]:
z = matriz_confusao_teste_nb

x = ['Predito Negativo', 'Predito Positivo']
y = ['Verdadeiro Negativo', 'Verdadeiro Positivo']

z_mask = [[1 if i == j else 0 for j in range(z.shape[1])] for i in range(z.shape[0])]
# Use the actual confusion numbers as annotation text
annot = z.astype(str).tolist()
# Colors: errors -> red, correct -> green
colorscale = [[0, "#d03d2a"], [1, "#73e526"]]

fig = ff.create_annotated_heatmap(
   z=z_mask,
   annotation_text=annot,
   
   x=x,
   y=y,
   colorscale=colorscale,
   showscale=False
)
# Make annotation text white for readability on colored cells
for a in fig.layout.annotations:
   a.font = dict(color='black', size=14)

fig.update_layout(title='Matriz de Confusão - Naive Bayes - teste', title_x=0.5 )


fig.show()

In [ ]:
# Relatório de classificação (teste)
# Precisão: De todas as previsões positivas, quantas estavam corretas
# Recall: De todos os positivos reais, quantos foram previstos corretamente
# F1-Score: Média harmônica entre precisão e recall
# Seu valor varia entre 0 e 1, quanto mais próximo de 1, melhor o modelo
# Support: Número de ocorrências de cada classe no conjunto de dados
relatorio_classificacao_teste = classification_report(y_teste, previsoes_teste_naive)
print(f'Relatório de Classificação (teste):\n{relatorio_classificacao_teste}')

##### **Análise de Treino**

In [ ]:
previsoes_treino_naive = naive.predict(x_treino)
previsoes_treino_naive

In [ ]:
acuracia_treino_naive = accuracy_score(y_treino, previsoes_treino_naive)
acuracia_treino_naive

In [ ]:
print(f'Acurácia do Naive Bayes (treino) é de: {acuracia_treino_naive*100:.2f}%')

In [ ]:
# Matriz de confusão de treino. Mostra os verdadeiros positivos, verdadeiros negativos, falsos positivos e falsos negativos
matriz_confusao_treino_nb = confusion_matrix(y_treino, previsoes_treino_naive)
print(f'Matriz de Confusão:\n{matriz_confusao_treino_nb}')
print(f'{matriz_confusao_treino_nb[0,0]} true negatives\n{matriz_confusao_treino_nb[0,1]} false positives\n{matriz_confusao_treino_nb[1,0]} false negatives\n{matriz_confusao_treino_nb[1,1]} true positives')

In [ ]:
z = matriz_confusao_treino_nb

z_mask = [[1 if i == j else 0 for j in range(z.shape[1])] for i in range(z.shape[0])]
# Use the actual confusion numbers as annotation text
annot = z.astype(str).tolist()
# Colors: errors -> red, correct -> green
colorscale = [[0, "#d03d2a"], [1, "#73e526"]]

fig = ff.create_annotated_heatmap(
   z=z_mask,
   annotation_text=annot,
   x=x,
   y=y,
   colorscale=colorscale,
   showscale=False
)
# Make annotation text white for readability on colored cells
for a in fig.layout.annotations:
   a.font = dict(color='black', size=14)
   
fig.show()

In [ ]:
# Relatório de classificação (treino)
# Precisão: De todas as previsões positivas, quantas estavam corretas
# Recall: De todos os positivos reais, quantos foram previstos corretamente
# F1-Score: Média harmônica entre precisão e recall
# Seu valor varia entre 0 e 1, quanto mais próximo de 1, melhor o modelo
# Support: Número de ocorrências de cada classe no conjunto de dados
relatorio_classificacao_treino = classification_report(y_treino, previsoes_treino_naive)
print(f'Relatório de Classificação (teste):\n{relatorio_classificacao_treino}')

#### **Validação Cruzada**

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

In [ ]:
# Separando os dados em 30 partes (folds)
# n_splits: número de partes que os dados serão divididos
# shuffle: embaralha os dados antes de dividir em folds
# random_state: garante que a aleatoriedade seja reproduzível
kfold = KFold(n_splits=30, shuffle=True, random_state=5)

In [ ]:
# Criando o modelo
modelo = GaussianNB()
resultado = cross_val_score(modelo, previsores3_esc, target, cv=kfold)

# Usamos a média dos resultados obtidos em cada iteração e o desvio padrão
print(f'Accuracy médio: {resultado.mean()*100:.2f}%')
print(f'Desvio Padrão: {resultado.std():.2f}')
acuracia_naive = min([acuracia_treino_naive, acuracia_teste])
# Naive Bayes parece ser um bom modelo para esse conjunto de dados, considerando a acurácia obtida tanto no conjunto de treino quanto no conjunto de teste, além dos resultados da validação cruzada. Previsor usado é o previsores3_esc que contém 20 atributos (após o one-hot encoding e padronização).
print(f'{acuracia_naive*100:.2f}% [Treino e Teste] e {resultado.mean()*100:.2f}% [Validação Cruzada] - com  {previsor_usado.shape[1]} atributos')

## **Máquina de Vetores de Suporte (SVM)**


[Documentação do SVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html)

---

Aplicado em problemas de aprendizagem supervisionada, tanto regressão como classficação. Em classificação é conhecido como **Classificador de Vetor de Suporte (SVC)**.   
Cria hiperplanos de separação entre os dados. Pode ser aplicado em problemas com dados linearmente e não linearmente separáveis.

---

🧠 **Mini-Resumo: Support Vector Classifier (SVC):**

O **SVC**, ou **Support Vector Classifier**, é a forma de **classificação** do algoritmo **Support Vector Machine (SVM)**.
Ele busca um **hiperplano ótimo** que separa as classes com a **maior margem possível**, ou seja, a distância entre o hiperplano e os pontos mais próximos de cada classe (os chamados *vetores de suporte*).

---

⚙️ **Conceito Matemático**

A ideia central é encontrar o hiperplano que satisfaz:

$$
w \cdot x + b = 0
$$

onde:

* Vetor de pesos (define a orientação do hiperplano):
$$
w
$$
* Vetor de características:
$$
x
$$
* Viés (bias), que desloca o hiperplano:
$$
b
$$

---

🎯 **Função Objetivo**

O modelo procura **maximizar a margem**
$$
\frac{2}{||w||}
$$
O que equivale a **minimizar** 
$$
||w||^2
$$
Sujeita às restrições:

$$
y_i (w \cdot x_i + b) \geq 1, \quad \forall i
$$

onde 
$$
y_i \in {-1, 1}
$$
representa as classes.

---

🧮 **Classificação de um Novo Ponto**

A decisão é feita com base no sinal da função:

$$
f(x) = w \cdot x + b
$$

Se f(x) > 0, o ponto pertence a uma classe;  
se f(x) < 0, pertence à outra.

---

🌐 **Kernel Trick**

Quando os dados **não são linearmente separáveis**, o SVC usa o **truque do kernel** (*kernel trick*), que transforma o espaço original em um espaço de dimensão superior, onde a separação é possível.

Exemplos de funções kernel:

* Linear:
$$
K(x_i, x_j) = x_i \cdot x_j
$$
* Polinomial: 
$$
K(x_i, x_j) = (x_i \cdot x_j + 1)^d
$$
* RBF (Radial Basis Function):
$$
K(x_i, x_j) = \exp(-\gamma ||x_i - x_j||^2)
$$

---

💡 **Em resumo:**
O **SVC** busca o limite ideal entre classes, garantindo **máxima margem**, **robustez** e **boa generalização**, mesmo em espaços não lineares.

---

**APLICAÇÕES:**
* Classificação;
* Categorização de textos;
* Reconhecimento de imagens;
* Detecção facial;
* Detecção de anomalias;
* Reconhecimento de letras manuscritas.
---
**VANTAGENS**
* Não influenciado por valores discrepantes (outliers);
* Solução de problemas lineares e não lineares (kernel trick);
* Muito efetivo para datasets grandes;
* Consegue aprender com características não pertencentes aos dados.
---
**DESVANTAGENS**
* Difícil interpretação teórica devido a matemática complexa;
* Difícil visualização gráfica;
* Lento comparado a outros algoritmos;
* Deve-se ter grande cuidado com definições dos hiperparâmetros para evitar overfitting e underfitting

---
⚙️ **PARÂMETROS DO **`svc`****
| Parâmetro                     | Valor Padrão | Descrição                                                                                                                         |
| ----------------------------- | ------------ | --------------------------------------------------------------------------------------------------------------------------------- |
| **`C`**                       | `1.0`        | Controla o equilíbrio entre erro de treinamento e margem. Valores altos buscam menos erro (mas podem gerar overfitting).          |
| **`kernel`**                  | `'rbf'`      | Função usada para transformar os dados. Pode ser `'linear'`, `'poly'`, `'rbf'`, `'sigmoid'` ou uma função customizada.            |
| **`degree`**                  | `3`          | Grau do polinômio usado quando `kernel='poly'`. Ignorado para outros kernels.                                                     |
| **`gamma`**                   | `'scale'`    | Define quanto cada ponto de treinamento influencia. Pode ser `'scale'` (1 / (n_features * X.var())) ou `'auto'` (1 / n_features). |
| **`coef0`**                   | `0.0`        | Termo independente usado nos kernels `'poly'` e `'sigmoid'`. Controla a curvatura da fronteira.                                   |
| **`shrinking`**               | `True`       | Usa heurística de redução (mais rápido, geralmente sem afetar o resultado).                                                       |
| **`probability`**             | `False`      | Se `True`, ativa a estimativa de probabilidade (usa `predict_proba`), mas torna o treino mais lento.                              |
| **`tol`**                     | `1e-3`       | Tolerância para o critério de parada (quanto menor, mais preciso e demorado).                                                     |
| **`cache_size`**              | `200`        | Tamanho do cache (em MB) para armazenar cálculos do kernel.                                                                       |
| **`class_weight`**            | `None`       | Peso das classes. Pode usar `'balanced'` para ajustar automaticamente conforme a frequência das classes.                          |
| **`verbose`**                 | `False`      | Mostra mensagens detalhadas durante o treinamento (útil para depuração).                                                          |
| **`max_iter`**                | `-1`         | Número máximo de iterações. `-1` significa sem limite.                                                                            |
| **`decision_function_shape`** | `'ovr'`      | Estratégia para múltiplas classes: `'ovr'` (one-vs-rest) ou `'ovo'` (one-vs-one).                                                 |
| **`break_ties`**              | `False`      | Em problemas multi-classes com `decision_function_shape='ovr'`, desempata previsões com base na distância ao hiperplano.          |
| **`random_state`**            | `None`       | Controla a reprodutibilidade dos resultados quando `probability=True` ou com `shuffle`.                                           |


In [ ]:
from sklearn.svm import SVC

In [ ]:
svc = SVC(kernel='rbf', random_state=1, C=4)
svc.fit(x_treino, y_treino)

#### **Analisando o Algoritmo**

##### **Análise de Teste**

In [ ]:
previsoes_teste_svc = svc.predict(x_teste)
previsoes_teste_svc

In [ ]:
y_teste.values

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
print(f'Acurácia do SVC (teste) é de: {accuracy_score(y_teste, previsoes_teste_svc)*100:.2f}%')

In [ ]:
confusion_mtx_svc_teste = confusion_matrix(y_teste, previsoes_teste_svc)
print(f'Matriz de Confusão SVC:\n{confusion_mtx_svc_teste}')

In [ ]:
z = confusion_mtx_svc_teste

x = ['Predito Negativo', 'Predito Positivo']
y = ['Verdadeiro Negativo', 'Verdadeiro Positivo']

z_mask = [[1 if i == j else 0 for j in range(z.shape[1])] for i in range(z.shape[0])]
# Use the actual confusion numbers as annotation text
annot = z.astype(str).tolist()
# Colors: errors -> red, correct -> green
colorscale = [[0, "#d03d2a"], [1, "#73e526"]]

fig = ff.create_annotated_heatmap(
   z=z_mask,
   annotation_text=annot,
   
   x=x,
   y=y,
   colorscale=colorscale,
   showscale=False
)
# Make annotation text white for readability on colored cells
for a in fig.layout.annotations:
   a.font = dict(color='black', size=14)

fig.update_layout(title='Matriz de Confusão - SVC - teste', title_x=0.5 )


fig.show()

In [ ]:
print(classification_report(y_teste, previsoes_teste_svc))

##### **Análise de Treino**

In [ ]:
previsores_treino_svc = svc.predict(x_treino)
previsores_treino_svc

In [ ]:
print(f'Acurácia do SVC (treino) é de: {accuracy_score(y_treino, previsores_treino_svc)*100:.2f}%')

In [ ]:
confusion_mtx_svc_treino = confusion_matrix(y_treino, previsores_treino_svc)
print(f'Matriz de Confusão SVC:\n{confusion_mtx_svc_treino}')

In [ ]:
# GRAFICO DA MATRIZ DE CONFUSÃO

z = confusion_mtx_svc_treino

x = ['Predito Negativo', 'Predito Positivo']
y = ['Verdadeiro Negativo', 'Verdadeiro Positivo']

z_mask = [[1 if i == j else 0 for j in range(z.shape[1])] for i in range(z.shape[0])]
# Use the actual confusion numbers as annotation text
annot = z.astype(str).tolist()
# Colors: errors -> red, correct -> green
colorscale = [[0, "#d03d2a"], [1, "#73e526"]]

fig = ff.create_annotated_heatmap(
   z=z_mask,
   annotation_text=annot,
   
   x=x,
   y=y,
   colorscale=colorscale,
   showscale=False
)
# Make annotation text white for readability on colored cells
for a in fig.layout.annotations:
   a.font = dict(color='black', size=14)

fig.update_layout(title='Matriz de Confusão - SVC - teste', title_x=0.5 )


fig.show()

#### **Validação Cruzada**

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

In [ ]:
kfold = KFold(n_splits=30, shuffle=True, random_state=5)

In [ ]:
modelo_svc = SVC(kernel='rbf', random_state=1, C=4)
resultado_svc = cross_val_score(modelo_svc, previsores3_esc, target, cv=kfold)

In [ ]:
acuracia_svc = resultado_svc.mean()
print(f'Acurácia médio SVC: {resultado_svc.mean()*100:.2f}%')

## **Regressão Logística**

[Documentação da Regressão Logística](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)

---
🧠 **Mini-Resumo: Regressão Logística**

A **Regressão Logística** é um algoritmo de **classificação supervisionada** que estima a **probabilidade de uma amostra pertencer a uma classe**.
Apesar do nome, ela é usada **para classificação binária ou multiclasse**, não para regressão numérica.

---

**Conceito Matemático**

Ela transforma uma combinação linear de variáveis em uma probabilidade entre 0 e 1 usando a **função sigmoide**:

$$
P(y=1|X) = \sigma(w \cdot X + b) = \frac{1}{1 + e^{-(w \cdot X + b)}}
$$

onde:

* Vetor de pesos (coeficientes):
$$
w
$$
* Vetor de atributos:
$$
X
$$
* Intercepto (bias):
$$
b
$$
* Função logística que mapeia o valor para o intervalo [0, 1]:
$$
\sigma(z)
$$

---

🎯 **Decisão**

A classificação é feita com base em um **limiar (threshold)** — normalmente 0.5:

$$
\hat{y} =
\begin{cases}
1, & \text{se } P(y=1|X) \geq 0.5 \
0, & \text{caso contrário}
\end{cases}
$$

---

🔧 **Função de Custo**

A regressão logística minimiza o **erro logarítmico (log loss)**, dado por:

$$
J(w, b) = -\frac{1}{m} \sum_{i=1}^{m} [y^{(i)} \log(\hat{y}^{(i)}) + (1 - y^{(i)}) \log(1 - \hat{y}^{(i)})]
$$

Essa função penaliza previsões erradas com base na distância da probabilidade real — valores muito errados geram custo alto.

---

🧩 **Regularização**

Para evitar **overfitting**, é comum adicionar termos de **penalização** aos pesos:

* **L2 (Ridge)**: penaliza ( w² ), suavizando os coeficientes grandes;
* **L1 (Lasso)**: pode zerar coeficientes irrelevantes (faz seleção de variáveis);
* **Elastic Net**: mistura das duas.

---

🚀 **Vantagens**

* Simples e interpretável;
* Estima probabilidades, não apenas rótulos;
* Boa performance em dados lineares;
* Base teórica sólida e fácil de regularizar.

---

⚠️ **Limitações**

* Não captura relações **não lineares** sem transformação prévia dos dados;
* Sensível a **outliers**;
* Depende de escalonamento (normalização ou padronização dos atributos).

---

💡 **Em resumo:**

> A **Regressão Logística** é um modelo linear probabilístico que usa a **função sigmoide** para converter combinações lineares de variáveis em probabilidades, sendo uma das abordagens mais clássicas e robustas para **classificação supervisionada**.

---

⚙️ **Parâmetros do **`LogisticRegression`****

| Parâmetro               | Valor Padrão | Descrição                                                                                                          |
| ----------------------- | ------------ | ------------------------------------------------------------------------------------------------------------------ |
| **`penalty`**           | `'l2'`       | Tipo de regularização (penalização). Pode ser `'l1'`, `'l2'`, `'elasticnet'` ou `'none'`. Controla o sobreajuste.  |
| **`dual`**              | `False`      | Usa a formulação dual do problema. Só aplicável com `penalty='l2'` e `solver='liblinear'`.                         |
| **`tol`**               | `1e-4`       | Tolerância para o critério de parada (quanto menor, mais preciso e demorado).                                      |
| **`C`**                 | `1.0`        | Inverso da força de regularização (igual ao SVC). Valores menores → regularização mais forte.                      |
| **`fit_intercept`**     | `True`       | Se adiciona o termo de intercepto (bias) no modelo.                                                                |
| **`intercept_scaling`** | `1`          | Escala aplicada ao intercepto quando `solver='liblinear'` e `fit_intercept=True`.                                  |
| **`class_weight`**      | `None`       | Peso das classes. Pode ser `'balanced'` para ajustar automaticamente conforme a frequência das classes.            |
| **`random_state`**      | `None`       | Define a semente aleatória para reprodutibilidade.                                                                 |
| **`solver`**            | `'lbfgs'`    | Algoritmo de otimização. Pode ser `'newton-cg'`, `'lbfgs'`, `'liblinear'`, `'sag'` ou `'saga'`.                    |
| **`max_iter`**          | `100`        | Número máximo de iterações do otimizador. Aumente se o modelo não convergir.                                       |
| **`multi_class`**       | `'auto'`     | Estratégia para múltiplas classes: `'ovr'` (um contra todos) ou `'multinomial'`. `'auto'` escolhe automaticamente. |
| **`verbose`**           | `0`          | Nível de detalhamento das mensagens de saída. 0 = silencioso.                                                      |
| **`warm_start`**        | `False`      | Se `True`, reutiliza os coeficientes da última execução para acelerar o treino.                                    |
| **`n_jobs`**            | `None`       | Número de CPUs usadas em cálculos paralelos. `-1` usa todas as disponíveis.                                        |
| **`l1_ratio`**          | `None`       | Mistura entre L1 e L2 (somente com `penalty='elasticnet'` e `solver='saga'`). 0 = L2, 1 = L1.                      |

---

💡 **Resumo prático:**

* Para uso geral: `solver='lbfgs'`, `penalty='l2'`, `C=1.0`.
* Para dados **muito grandes**: `solver='saga'` (suporta regularização L1 e L2).
* Para **multiclasse real**: `multi_class='multinomial'`, `solver='lbfgs'`.
* Para **classes desbalanceadas**: `class_weight='balanced'`.

#### **Treinando Algoritmo**

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
logistica = LogisticRegression(solver='lbfgs', max_iter=200, random_state=1, penalty='l2', C=1, tol=0.0001)
logistica.fit(x_treino, y_treino)

#### **Avaliação do Algortimo**

##### **Análise de Teste**

In [ ]:
logistica.intercept_ # Coeficiente linear (bias), um valor para cada classe

In [ ]:
logistica.coef_ # Coeficientes dos atributos (pesos) para cada classe

In [ ]:
previsoes_teste_logistica = logistica.predict(x_teste)
previsoes_teste_logistica

In [ ]:
y_teste.values

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
print(f'Acurácia da Logística (teste) é de: {accuracy_score(y_teste, previsoes_teste_logistica)*100:.2f}%')

In [ ]:
confusion_mtx_logica_teste = confusion_matrix(y_teste, previsoes_teste_logistica)
confusion_mtx_logica_teste

In [ ]:
# GRAFICO DA MATRIZ DE CONFUSÃO

z = confusion_mtx_logica_teste
# GRAFICO DA MATRIZ DE CONFUSÃO

x = ['Predito Negativo', 'Predito Positivo']
y = ['Verdadeiro Negativo', 'Verdadeiro Positivo']

z_mask = [[1 if i == j else 0 for j in range(z.shape[1])] for i in range(z.shape[0])]
# Use the actual confusion numbers as annotation text
annot = z.astype(str).tolist()
# Colors: errors -> red, correct -> green
colorscale = [[0, "#d03d2a"], [1, "#73e526"]]

fig = ff.create_annotated_heatmap(
   z=z_mask,
   annotation_text=annot,
   
   x=x,
   y=y,
   colorscale=colorscale,
   showscale=False
)
# Make annotation text white for readability on colored cells
for a in fig.layout.annotations:
   a.font = dict(color='black', size=14)

fig.update_layout(title='Matriz de Confusão Teste - Regressão Logística', title_x=0.5 )


fig.show()

In [ ]:
print(classification_report(y_teste, previsoes_teste_logistica))

##### **Análise de Treino**

In [ ]:
previsoes_treino_logistica = logistica.predict(x_treino)
previsoes_treino_logistica

In [ ]:
print(f'Acurácia da Logística (treino) é de: {accuracy_score(y_treino, previsoes_treino_logistica)*100:.2f}%')

In [ ]:
confusion_mtx_logisca_treino = confusion_matrix(y_treino, previsoes_treino_logistica)
confusion_mtx_logisca_treino

In [ ]:
# GRAFICO DA MATRIZ DE CONFUSÃO

z = confusion_mtx_logisca_treino

x = ['Predito Negativo', 'Predito Positivo']
y = ['Verdadeiro Negativo', 'Verdadeiro Positivo']

z_mask = [[1 if i == j else 0 for j in range(z.shape[1])] for i in range(z.shape[0])]
# Use the actual confusion numbers as annotation text
annot = z.astype(str).tolist()
# Colors: errors -> red, correct -> green
colorscale = [[0, "#d03d2a"], [1, "#73e526"]]

fig = ff.create_annotated_heatmap(
   z=z_mask,
   annotation_text=annot,
   
   x=x,
   y=y,
   colorscale=colorscale,
   showscale=False
)
# Make annotation text white for readability on colored cells
for a in fig.layout.annotations:
   a.font = dict(color='black', size=14)

fig.update_layout(title='Matriz de Confusão Treino - Regressão Logística', title_x=0.5 )


fig.show()

#### **Validação Cruzada**

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

In [ ]:
kfold = KFold(n_splits=30, shuffle=True, random_state=5)

In [ ]:
modelo_logistica = LogisticRegression(solver='lbfgs', max_iter=200, random_state=1, penalty='l2', C=1, tol=0.0001)
resultado_logistica = cross_val_score(modelo_logistica, previsores3_esc, target, cv=kfold)
acuracia_logistica = resultado_logistica.mean()
print(f'Accuracy médio Logística: {acuracia_logistica*100:.2f}%')

## **K-Nearest Neighbors (KNN)**

[Documentação do KNN](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)

---
🧠 **Mini-Resumo: K-Nearest Neighbors**

O **K-Nearest Neighbors** é um algoritmo de **classificação e regressão** baseado em **instâncias**, que não cria um modelo explícito — ele decide com base nos **vizinhos mais próximos** de um novo ponto.

A ideia é simples:

> “Diga-me quem são seus vizinhos e eu direi quem você é.”

---

⚙️ **Conceito Matemático**

Para um novo ponto ( x ), o KNN calcula a **distância** entre ( x ) e todos os pontos do conjunto de treino, selecionando os **k** vizinhos mais próximos.

A classe é então decidida por **votação majoritária** (classificação) ou pela **média dos valores** (regressão):

$$
\hat{y} =
\begin{cases}
\text{modo}(y_{vizinhos}), & \text{para classificação} \
\frac{1}{k} \sum_{i=1}^{k} y_i, & \text{para regressão}
\end{cases}
$$

---

📏 **Distâncias Comuns**

* **Euclidiana:** $$d(x, x_i) = \sqrt{\sum_j (x_j - x_{ij})^2} $$
* **Manhattan:** $$d(x, x_i) = \sum_j |x_j - x_{ij}|$$
* **Minkowski:** generaliza as anteriores com um parâmetro ( p )$$d(x, x_i) = \sum_j(w * |x_j - x_{ij}|^p)^(1/p)$$

---

🚀 **Vantagens**

* Simples e intuitivo;
* Não faz suposições sobre a distribuição dos dados;
* Funciona bem em problemas com **fronteiras complexas**.

---

⚠️ **Limitações**

* Lento para grandes bases (precisa comparar com todos os pontos);
* Sensível à escala dos atributos (necessita normalização);
* Desempenho depende fortemente do **k** escolhido (tentativa e erro);
* Necessita transformar dados categóricos em numéricos.

---

🧩 **Escolha do k**

* **k pequeno** → modelo mais sensível, sujeito a ruído (*overfitting*);
* **k grande** → modelo mais suave, porém pode ignorar padrões locais (*underfitting*).

---

🧮 **Fórmula da Ponderação (quando usada)**

Quando `weights='distance'`, vizinhos mais próximos têm maior influência:

$$
w_i = \frac{1}{d(x, x_i)}
$$

---

⚙️ **Hiperparâmetros do **`KNeighborsClassifier`****

| Parâmetro           | Valor Padrão  | Descrição                                                                                                                    |
| ------------------- | ------------- | ---------------------------------------------------------------------------------------------------------------------------- |
| **`n_neighbors`**   | `5`           | Número de vizinhos a considerar para fazer a predição.                                                                       |
| **`weights`**       | `'uniform'`   | Define o peso dos vizinhos. `'uniform'` = todos iguais; `'distance'` = mais próximos têm mais peso.                          |
| **`algorithm`**     | `'auto'`      | Método usado para encontrar vizinhos. Pode ser `'ball_tree'`, `'kd_tree'`, `'brute'` ou `'auto'` (deixa o sklearn escolher). |
| **`leaf_size`**     | `30`          | Tamanho da folha para algoritmos baseados em árvores (impacta tempo de busca e memória).                                     |
| **`p`**             | `2`           | Parâmetro da **distância de Minkowski**. `p=1` → Manhattan, `p=2` → Euclidiana.                                              |
| **`metric`**        | `'minkowski'` | Métrica de distância usada para calcular vizinhos. Pode ser `'euclidean'`, `'manhattan'`, `'chebyshev'`, etc.                |
| **`metric_params`** | `None`        | Parâmetros adicionais para métricas personalizadas.                                                                          |
| **`n_jobs`**        | `None`        | Número de CPUs usadas em paralelo. `-1` usa todas disponíveis.                                                               |

---

💡 **Resumo prático:**

> O KNN é simples, eficaz e não precisa de treinamento, mas deve ser usado com cuidado em **dados grandes** ou **sem normalização**.
> Ajustar `k`, `weights` e `p` costuma ser essencial para obter o melhor desempenho.

#### **Treinando Algoritmo**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5, weights='uniform', metric='minkowski', p=2)
knn.fit(x_treino, y_treino)

#### **Análise do Algoritmo**

##### **Análise de Teste**

In [ ]:
previsoes_knn_teste = knn.predict(x_teste)
previsoes_knn_teste

In [ ]:
y_teste.values

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
print(f'Acurácia do KNN (teste) é de: {accuracy_score(y_teste, previsoes_knn_teste)*100:.2f}%')

In [ ]:
confusion_matrix_knn_teste = confusion_matrix(y_teste, previsoes_knn_teste)
confusion_matrix_knn_teste

In [ ]:
# GRAFICO DA MATRIZ DE CONFUSÃO

z = confusion_matrix_knn_teste

x = ['Predito Negativo', 'Predito Positivo']
y = ['Verdadeiro Negativo', 'Verdadeiro Positivo']

z_mask = [[1 if i == j else 0 for j in range(z.shape[1])] for i in range(z.shape[0])]
# Use the actual confusion numbers as annotation text
annot = z.astype(str).tolist()
# Colors: errors -> red, correct -> green
colorscale = [[0, "#d03d2a"], [1, "#73e526"]]

fig = ff.create_annotated_heatmap(
   z=z_mask,
   annotation_text=annot,
   
   x=x,
   y=y,
   colorscale=colorscale,
   showscale=False
)
# Make annotation text white for readability on colored cells
for a in fig.layout.annotations:
   a.font = dict(color='black', size=14)

fig.update_layout(title='Matriz de Confusão Teste - K-Nearest Neighbors', title_x=0.5 )


fig.show()

In [ ]:
print(classification_report(y_teste, previsoes_knn_teste))

##### **Análise do Treino**

In [ ]:
previsoes_knn_treino = knn.predict(x_treino)
previsoes_knn_treino

In [ ]:
y_treino.values

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
print(f'Acurácia do KNN (treino) é de: {accuracy_score(y_treino, previsoes_knn_treino)*100:.2f}%')

In [ ]:
confusion_matrix_knn_treino = confusion_matrix(y_treino, previsoes_knn_treino)
confusion_matrix_knn_treino

In [ ]:
# GRAFICO DA MATRIZ DE CONFUSÃO

z = confusion_matrix_knn_treino

x = ['Predito Negativo', 'Predito Positivo']
y = ['Verdadeiro Negativo', 'Verdadeiro Positivo']

z_mask = [[1 if i == j else 0 for j in range(z.shape[1])] for i in range(z.shape[0])]
# Use the actual confusion numbers as annotation text
annot = z.astype(str).tolist()
# Colors: errors -> red, correct -> green
colorscale = [[0, "#d03d2a"], [1, "#73e526"]]

fig = ff.create_annotated_heatmap(
   z=z_mask,
   annotation_text=annot,
   
   x=x,
   y=y,
   colorscale=colorscale,
   showscale=False
)
# Make annotation text white for readability on colored cells
for a in fig.layout.annotations:
   a.font = dict(color='black', size=14)

fig.update_layout(title='Matriz de Confusão Treino - K-Nearest Neighbors', title_x=0.5 )


fig.show()

In [ ]:
print(classification_report(y_treino, previsoes_knn_treino))

#### **Validação Cruzada**

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

In [ ]:
kfold = KFold(n_splits=30, shuffle=True, random_state=5)

In [ ]:
modelo_knn = KNeighborsClassifier(n_neighbors=5, weights='uniform', metric='minkowski', p=2)
resultado_knn = cross_val_score(modelo_knn, previsores3_esc, target, cv=kfold)
acuracia_knn = resultado_knn.mean()
print(f'Accuracy médio KNN: {acuracia_knn*100:.2f}%')

## **Ávores de Decisão**

[Documentação Árvore de Decisão](https://scikit-learn.org/stable/modules/tree.html)

---
🧠 **Mini-Resumo: Árvores de Decisão (Decision Tree)**

As **árvores de decisão** são algoritmos de **classificação e regressão** que aprendem **regras hierárquicas** a partir dos dados, dividindo-os em ramos até chegar a **decisões finais (folhas)**.
Cada **nó interno** representa uma condição sobre uma variável, e cada **ramo** é o resultado dessa condição.

---

🌿 **Conceito Básico**

A árvore divide recursivamente o conjunto de dados em subconjuntos **cada vez mais homogêneos**, com base em medidas de **impureza**.

* **Índice de Gini:**

$$
Gini = 1 - \sum_{i=1}^{C} p_i^2
$$

* **Entropia:**

$$
Entropy = - \sum_{i=1}^{C} p_i \log_2(p_i)
$$

onde ( p_i ) é a proporção de exemplos da classe ( i ) no nó.

O algoritmo escolhe a divisão que **maximiza o ganho de informação**:

$$
Gain = Impureza_{pai} - \sum_{filhos} \frac{N_{filho}}{N_{pai}} \times Impureza_{filho}
$$

---

🎯 **Vantagens**

* Fácil de interpretar e visualizar;
* Suporta dados numéricos e categóricos;
* Pouco pré-processamento (não exige normalização);
* Captura relações não lineares entre variáveis;
* Trabalha com valores faltantes, variáveis categóricas e numéricas.

---

⚠️ **Limitações**

* Tende a **overfitting** (cresce demais e memoriza o treino);
* Pequenas mudanças nos dados podem gerar estruturas diferentes;
* Não garante a construção da melhor estrutura para os dados de treino em questão (necessita treinar várias árvores distintas);
* Menos robusta que métodos baseados em múltiplas árvores (como Random Forest).

---

✂️ **Poda (Pruning)**

A **poda** é uma técnica essencial para **controlar o crescimento da árvore** e **reduzir o overfitting**.
Ela remove ramos desnecessários que não contribuem significativamente para o desempenho, melhorando a **generalização**.

Existem dois tipos principais:

1. Poda Prévia (*Pre-pruning*)

A árvore é **interrompida durante o crescimento**, com base em critérios como:

* **`max_depth`** → profundidade máxima permitida;
* **`min_samples_split`** → mínimo de amostras para dividir um nó;
* **`min_samples_leaf`** → mínimo de amostras em cada folha;
* **`min_impurity_decrease`** → divisão ocorre apenas se reduzir impureza acima desse valor.

👉 Evita crescimento excessivo **desde o início**.

---

2. Poda Pós-treinamento (*Post-pruning* ou *Cost Complexity Pruning*)

A árvore é **crescida até o fim** e depois **reduzida**, removendo ramos com **baixo ganho de informação em relação ao custo de complexidade**.
O controle é feito com o parâmetro **`ccp_alpha`** (*Cost-Complexity Pruning Alpha*).

A ideia é minimizar a seguinte função:

$$
R_\alpha(T) = R(T) + \alpha \cdot |T|
$$

onde:

* Erro de classificação da árvore ( T ):
$$
\R(T)
$$   
* Número de folhas:
$$
|T|
$$
* Parâmetro de penalização (quanto maior, mais poda):
$$
\alpha
$$

💡 Valores maiores de ( \alpha ) → árvores menores e mais simples.
O sklearn oferece o método **`cost_complexity_pruning_path()`** para visualizar os valores ideais de
$$
\alpha
$$

---

🧩 **Exemplo de Regra Simples**

```text
Se (idade <= 25) e (salário > 3000):
    classe = "Aprovado"
Senão:
    classe = "Reprovado"
```

---

⚙️ **Hiperparâmetros do **`DecisionTreeClassifier`****

| Parâmetro                      | Valor Padrão | Descrição                                                                                             |
| ------------------------------ | ------------ | ----------------------------------------------------------------------------------------------------- |
| **`criterion`**                | `'gini'`     | Função usada para medir a qualidade da divisão (`'gini'` ou `'entropy'`).                             |
| **`splitter`**                 | `'best'`     | Estratégia para escolher a divisão: `'best'` (melhor) ou `'random'`.                                  |
| **`max_depth`**                | `None`       | Profundidade máxima da árvore (poda prévia).                                                          |
| **`min_samples_split`**        | `2`          | Mínimo de amostras necessário para dividir um nó (poda prévia).                                       |
| **`min_samples_leaf`**         | `1`          | Mínimo de amostras exigido em uma folha (poda prévia).                                                |
| **`min_weight_fraction_leaf`** | `0.0`        | Fração mínima de peso das amostras em uma folha (poda prévia).                                        |
| **`max_features`**             | `None`       | Número de atributos considerados para cada divisão.                                                   |
| **`random_state`**             | `None`       | Define a semente aleatória para reprodutibilidade.                                                    |
| **`max_leaf_nodes`**           | `None`       | Número máximo de folhas permitidas (poda prévia).                                                     |
| **`min_impurity_decrease`**    | `0.0`        | Divide o nó apenas se a impureza diminuir acima desse valor.                                          |
| **`class_weight`**             | `None`       | Ajusta pesos das classes para bases desbalanceadas.                                                   |
| **`ccp_alpha`**                | `0.0`        | Parâmetro de **poda pós-treinamento** por custo-complexidade (valores maiores geram árvores menores). |

---

💡 **Em resumo:**

> As **árvores de decisão** são modelos transparentes e poderosos, mas **precisam de poda** para equilibrar **precisão e generalização**.
> A **poda prévia** controla o crescimento durante o treinamento, e a **poda pós-treinamento** refina a árvore removendo divisões de baixo ganho.

---

💡 **Resumo prático:**

> As **árvores de decisão** são modelos interpretáveis, flexíveis e poderosos, mas precisam de **regularização (como profundidade máxima ou poda)** para evitar **overfitting**.
> São a base de modelos ensemble mais robustos como **Random Forest** e **Gradient Boosting**.

#### **Treinando Algoritmo**

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
arvore = DecisionTreeClassifier(criterion='gini', random_state=0, max_depth=5, min_samples_split=2)
arvore.fit(x_treino, y_treino)


##### **Análise do Algoritmo**

##### **Análise de teste**

In [ ]:
previsoes_arvore_teste = arvore.predict(x_teste)
previsoes_arvore_teste

In [ ]:
y_teste.values

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
print(f'Acurácia de Arvore de Decisão (teste) é de: {accuracy_score(y_teste, previsoes_arvore_teste)*100:.2f}%')

In [ ]:
confusion_mtx_tree_teste = confusion_matrix(y_teste, previsoes_arvore_teste)
confusion_mtx_tree_teste

In [ ]:
# GRAFICO DA MATRIZ DE CONFUSÃO

z = confusion_mtx_tree_teste

x = ['Predito Negativo', 'Predito Positivo']
y = ['Verdadeiro Negativo', 'Verdadeiro Positivo']

z_mask = [[1 if i == j else 0 for j in range(z.shape[1])] for i in range(z.shape[0])]
# Use the actual confusion numbers as annotation text
annot = z.astype(str).tolist()
# Colors: errors -> red, correct -> green
colorscale = [[0, "#d03d2a"], [1, "#73e526"]]

fig = ff.create_annotated_heatmap(
   z=z_mask,
   annotation_text=annot,
   
   x=x,
   y=y,
   colorscale=colorscale,
   showscale=False
)
# Make annotation text white for readability on colored cells
for a in fig.layout.annotations:
   a.font = dict(color='black', size=14)

fig.update_layout(title='Matriz de Confusão Teste - Arvore de Decisão', title_x=0.5 )


fig.show()

In [ ]:
print(classification_report(y_teste,previsoes_arvore_teste))

##### **Análise de Treino**

In [ ]:
previsoes_treino_tree = arvore.predict(x_treino)
previsoes_treino_tree

In [ ]:
y_treino.values

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
print(f'Acurácia de Arvore de Decisão (treino) é de: {accuracy_score(y_treino, previsoes_treino_tree)*100:.2f}%')

In [ ]:
confusion_mtx_tree_treino = confusion_matrix(y_treino, previsoes_treino_tree)
confusion_mtx_tree_treino

In [ ]:
# GRAFICO DA MATRIZ DE CONFUSÃO

z = confusion_mtx_tree_treino

x = ['Predito Negativo', 'Predito Positivo']
y = ['Verdadeiro Negativo', 'Verdadeiro Positivo']

z_mask = [[1 if i == j else 0 for j in range(z.shape[1])] for i in range(z.shape[0])]
# Use the actual confusion numbers as annotation text
annot = z.astype(str).tolist()
# Colors: errors -> red, correct -> green
colorscale = [[0, "#d03d2a"], [1, "#73e526"]]

fig = ff.create_annotated_heatmap(
   z=z_mask,
   annotation_text=annot,
   
   x=x,
   y=y,
   colorscale=colorscale,
   showscale=False
)
# Make annotation text white for readability on colored cells
for a in fig.layout.annotations:
   a.font = dict(color='black', size=14)

fig.update_layout(title='Matriz de Confusão Treino - Arvore de Decisão', title_x=0.5 )


fig.show()

#### **Validação Cruzada**

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

In [ ]:
kfold = KFold(n_splits=30, shuffle=True, random_state=5)
modelo_tree = DecisionTreeClassifier(criterion='gini', random_state=0, max_depth=5, min_samples_split=2)
resultado_tree = cross_val_score(modelo_tree, previsores3_esc, target, cv=kfold)
acuracia_tree = resultado_tree.mean()
print(f'Accuracy médio Arvore de Decisão: {acuracia_tree*100:.2f}%')

## **Random Forest**

[Documentação Random Forest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)

---
🧠 **Mini-Resumo: Random Forest (Floresta Aleatória)**

O **Random Forest** é um algoritmo de **ensemble** baseado em **múltiplas Árvores de Decisão**, criado para melhorar a **precisão** e **robustez** das árvores individuais.

Ele combina o conceito de:

> 🌳 **“muitos modelos fracos (árvores) → um modelo forte (floresta)”**

---

⚙️ **Conceito Matemático**

A ideia é treinar **várias árvores de decisão** em **amostras aleatórias dos dados** e combinar seus resultados.

1. Cada árvore é treinada com uma **amostra bootstrap** (amostragem com reposição);
2. Em cada divisão, apenas um **subconjunto aleatório de atributos** é considerado;
3. A predição final é feita por **votação majoritária** (classificação) ou **média** (regressão).

🧮 **Classificação**

$$
\hat{y} = \text{modo}{h_1(x), h_2(x), ..., h_T(x)}
$$

🧮 **Regressão**

$$
\hat{y} = \frac{1}{T} \sum_{t=1}^{T} h_t(x)
$$

onde
$$
h_t(x)
$$
é a predição da árvore *t* e *T* é o número total de árvores.

---

🌐 **Características-Chave**

* Reduz **overfitting** presente em árvores únicas;
* Fornece **importância das variáveis**;
* Trabalha bem com **grandes volumes de dados**;
* É um método **não paramétrico** (não assume distribuição dos dados).

---

🎯 **Vantagens**

* Alta precisão e robustez;
* Menor risco de overfitting;
* Funciona bem com dados mistos (numéricos e categóricos);
* Funciona bem com dados faltantes;
* Mede a **importância das features** (`feature_importances_`).

---

⚠️ **Limitações**

* Menos interpretável que uma árvore individual;
* Pode ser **mais lento** em bases grandes;
* Requer mais memória;
* Difícil de visualizar.

---

🌲 **Poda Implícita**

Diferente das árvores simples, o **Random Forest não precisa de poda explícita**.
Cada árvore tende a crescer até o máximo, mas o conjunto é **regularizado pela aleatoriedade** e pela **média dos resultados**, o que naturalmente controla o overfitting.

---

📊 **Importância das Variáveis**

O modelo calcula a importância de cada atributo com base na **redução média da impureza** causada pelas divisões em todas as árvores.

$$
Importance(f_j) = \frac{1}{T} \sum_{t=1}^{T} \sum_{n \in N_j} \frac{N_n}{N} \Delta Impureza_n
$$

onde:

* Nós onde a variável ( f_j ) foi usada:
$$
N_j
$$
* Redução da impureza no nó ( n ):
$$
\Delta Impureza_n
$$
* Número de amostras no nó:
$$
N_n
$$
* Total de amostras:
$$
N
$$

---

⚙️ **Hiperparâmetros do **`RandomForestClassifier`****

| Parâmetro                      | Valor Padrão | Descrição                                                                                                      |
| ------------------------------ | ------------ | -------------------------------------------------------------------------------------------------------------- |
| **`n_estimators`**             | `100`        | Número de árvores na floresta. Aumentar melhora o desempenho, mas aumenta o tempo de treino.                   |
| **`criterion`**                | `'gini'`     | Função para medir a qualidade da divisão (`'gini'` ou `'entropy'`).                                            |
| **`max_depth`**                | `None`       | Profundidade máxima das árvores. Controla overfitting.                                                         |
| **`min_samples_split`**        | `2`          | Número mínimo de amostras para dividir um nó.                                                                  |
| **`min_samples_leaf`**         | `1`          | Mínimo de amostras exigido em uma folha.                                                                       |
| **`min_weight_fraction_leaf`** | `0.0`        | Fração mínima do peso total das amostras em uma folha.                                                         |
| **`max_features`**             | `'sqrt'`     | Número de atributos considerados em cada divisão (`'sqrt'` para classificação, `'log2'` ou um número inteiro). |
| **`max_leaf_nodes`**           | `None`       | Número máximo de folhas permitido.                                                                             |
| **`min_impurity_decrease`**    | `0.0`        | Divide o nó apenas se a impureza diminuir acima desse valor.                                                   |
| **`bootstrap`**                | `True`       | Se `True`, usa amostragem com reposição para criar cada árvore.                                                |
| **`oob_score`**                | `False`      | Ativa o uso de amostras fora do bootstrap (*out-of-bag*) para validação interna.                               |
| **`n_jobs`**                   | `None`       | Número de núcleos usados no processamento paralelo (`-1` usa todos).                                           |
| **`random_state`**             | `None`       | Semente aleatória para reprodutibilidade.                                                                      |
| **`verbose`**                  | `0`          | Nível de detalhamento do processo de treino.                                                                   |
| **`warm_start`**               | `False`      | Se `True`, adiciona mais árvores sem reiniciar o modelo.                                                       |
| **`class_weight`**             | `None`       | Ajusta pesos das classes (`'balanced'` para desbalanceamento).                                                 |
| **`ccp_alpha`**                | `0.0`        | Parâmetro de poda de custo-complexidade aplicado a cada árvore individual.                                     |
| **`max_samples`**              | `None`       | Proporção ou número de amostras usadas para treinar cada árvore (se `bootstrap=True`).                         |

---

💡 **Em resumo:**

> O **Random Forest** é uma combinação de várias árvores de decisão independentes, cada uma treinada com amostras e atributos aleatórios.
> Essa diversidade torna o modelo **preciso, estável e generalizável**, reduzindo drasticamente o risco de overfitting.
> É um dos algoritmos mais **versáteis e poderosos** da aprendizagem supervisionada.

#### **Treinando Algoritmo**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
random = RandomForestClassifier(n_estimators=100, criterion='gini', random_state=0, max_depth=5, min_samples_split=2)
random.fit(x_treino, y_treino)

#### **Análise do Algoritmo**

##### **Análise de teste**

In [ ]:
random_teste = random.predict(x_teste)
random_teste

In [ ]:
y_teste.values

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
print(f'Acurácia do Random Forest (teste) é de: {accuracy_score(y_teste, random_teste)*100:.2f}%')

In [ ]:
confusion_matrix_randon_teste =  confusion_matrix(y_teste, random_teste)
confusion_matrix_randon_teste

In [ ]:
# GRAFICO DA MATRIZ DE CONFUSÃO

z = confusion_matrix_randon_teste

x = ['Predito Negativo', 'Predito Positivo']
y = ['Verdadeiro Negativo', 'Verdadeiro Positivo']

z_mask = [[1 if i == j else 0 for j in range(z.shape[1])] for i in range(z.shape[0])]
# Use the actual confusion numbers as annotation text
annot = z.astype(str).tolist()
# Colors: errors -> red, correct -> green
colorscale = [[0, "#d03d2a"], [1, "#73e526"]]

fig = ff.create_annotated_heatmap(
   z=z_mask,
   annotation_text=annot,
   
   x=x,
   y=y,
   colorscale=colorscale,
   showscale=False
)
# Make annotation text white for readability on colored cells
for a in fig.layout.annotations:
   a.font = dict(color='black', size=14)

fig.update_layout(title='Matriz de Confusão Teste - Randon Forest', title_x=0.5 )


fig.show()

In [ ]:
classification_report_random = classification_report(y_teste, random_teste)
print(f'Relatório de Classificação (teste):\n{classification_report_random}')

##### **Análise de treino**

In [ ]:
random_treino = random.predict(x_treino)
random_treino

In [ ]:
y_treino.values

In [ ]:
print(f'Acurácia do Random Forest (treino) é de: {accuracy_score(y_treino, random_treino)*100:.2f}%')

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
confusion_matrix_randon_treino = confusion_matrix(y_treino, random_treino)
confusion_matrix_randon_treino

In [ ]:
# GRAFICO DA MATRIZ DE CONFUSÃO

z = confusion_matrix_randon_treino
x = ['Predito Negativo', 'Predito Positivo']
y = ['Verdadeiro Negativo', 'Verdadeiro Positivo']

z_mask = [[1 if i == j else 0 for j in range(z.shape[1])] for i in range(z.shape[0])]
# Use the actual confusion numbers as annotation text
annot = z.astype(str).tolist()
# Colors: errors -> red, correct -> green
colorscale = [[0, "#d03d2a"], [1, "#73e526"]]

fig = ff.create_annotated_heatmap(
   z=z_mask,
   annotation_text=annot,
   
   x=x,
   y=y,
   colorscale=colorscale,
   showscale=False
)
# Make annotation text white for readability on colored cells
for a in fig.layout.annotations:
   a.font = dict(color='black', size=14)

fig.update_layout(title='Matriz de Confusão Treino - Randon Forest', title_x=0.5 )


fig.show()

In [ ]:
classification_report_random = classification_report(y_treino, random_treino)
print(f'Relatório de Classificação (teste):\n{classification_report_random}')

#### **Validação Cruzada**

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

In [ ]:
kfold = KFold(n_splits=30, shuffle=True, random_state=0)
modelo_random = RandomForestClassifier(n_estimators=100, criterion='gini', random_state=0, max_depth=5, min_samples_split=2)
resultado_random = cross_val_score(modelo_random, previsores3_esc, target, cv =kfold)
acuracia_random = resultado_random.mean()
print(f'Accuracy médio Random Forest: {acuracia_random*100:.2f}%')

## **XGBoost**


[Documentação XGBoost](https://xgboost.readthedocs.io/en/stable/)

---
🧠 **Mini-resumo: XGBoost (Extreme Gradient Boosting)**

O **XGBoost** é um algoritmo de **ensemble baseado em árvores de decisão**, desenvolvido como uma implementação otimizada do **Gradient Boosting** que por sua vez é uma evolução do **Random Forest**.
Ele combina **velocidade, regularização e paralelismo**, sendo um dos algoritmos mais poderosos e populares em aprendizado supervisionado, tanto de classificação quanto de regressão.  

---

⚙️ **Conceito Matemático**

O XGBoost constrói o modelo **de forma aditiva**, criando árvores **sequencialmente**.
Cada nova árvore tenta **corrigir os erros** (resíduos) das anteriores, otimizando uma função de perda por **gradiente descendente**.

A predição final é a soma das predições das árvores:

$$
\hat{y}*i = \sum*{k=1}^{K} f_k(x_i), \quad f_k \in \mathcal{F}
$$

onde 
$$
f_k(x)
$$
representa cada árvore de decisão e 
$$
\mathcal{F}
$$
é o espaço de funções possíveis (árvores).

A função de custo é composta por **dois termos**:

$$
Obj = \sum_{i=1}^{n} l(y_i, \hat{y}*i) + \sum*{k=1}^{K} \Omega(f_k)
$$

onde:

* função de perda (ex: log loss, MSE):
$$ 
l(y_i, \hat{y}_i)
$$
* termo de regularização que penaliza árvores complexas (T = número de folhas):
$$ 
\Omega(f_k) = \gamma T + \frac{1}{2}\lambda ||w||^2
$$

---

🎯 **Características-Chave**

* Usa **boosting** (adição sequencial de árvores para reduzir erro);
* Faz **regularização L1 e L2** para evitar overfitting;
* Suporta **paralelismo** e **cache inteligente** (muito rápido);
* Possui suporte nativo a **dados faltantes (NaN)**;
* Inclui **early stopping**, **importância de features**, e **poda inteligente**.

---

🚀 **Vantagens**

* Extremamente rápido e eficiente;
* Alta precisão e generalização;
* Controle fino de regularização;
* Suporte a classificação, regressão e ranking;
* Funciona bem mesmo sem normalização de dados.

---

⚠️ **Limitações**

* Muitos hiperparâmetros (requer tuning cuidadoso);
* Pode sobreajustar se o *learning rate* for alto;
* Menos interpretável que uma árvore única.

---

🌳 **Diferença para Random Forest**

| Aspecto       | Random Forest                  | XGBoost                                               |
| ------------- | ------------------------------ | ----------------------------------------------------- |
| Treinamento   | Árvores **independentes**      | Árvores **sequenciais** (cada uma corrige a anterior) |
| Combinação    | Votação ou média               | Soma ponderada (boosting)                             |
| Regularização | Implícita (pela aleatoriedade) | Explícita (L1 e L2)                                   |
| Overfitting   | Menor risco                    | Requer controle do *learning rate*                    |
| Velocidade    | Boa                            | Excelente (graças a otimizações)                      |

---

⚙️ **Principais Hiperparâmetros do **`XGBClassifier`****

| Parâmetro                   | Valor Padrão        | Descrição                                                                                                             |
| --------------------------- | ------------------- | --------------------------------------------------------------------------------------------------------------------- |
| **`n_estimators`**          | `100`               | Número de árvores (iterações de boosting).                                                                            |
| **`learning_rate`**         | `0.1`               | Taxa de aprendizado. Controla o peso de cada nova árvore. Valores menores aumentam precisão, mas exigem mais árvores. |
| **`max_depth`**             | `6`                 | Profundidade máxima das árvores. Controla complexidade e overfitting.                                                 |
| **`min_child_weight`**      | `1`                 | Peso mínimo total das amostras em um nó folha. Valores maiores tornam o modelo mais conservador.                      |
| **`gamma`**                 | `0`                 | Redução mínima da perda exigida para dividir um nó. Funciona como regularizador.                                      |
| **`subsample`**             | `1`                 | Fração das amostras usadas para treinar cada árvore. Menor valor → menos overfitting.                                 |
| **`colsample_bytree`**      | `1`                 | Fração das features usadas para construir cada árvore.                                                                |
| **`colsample_bylevel`**     | `1`                 | Fração das features usadas em cada nível de uma árvore.                                                               |
| **`colsample_bynode`**      | `1`                 | Fração das features usadas em cada divisão.                                                                           |
| **`reg_lambda`**            | `1`                 | Regularização L2 aplicada aos pesos das folhas.                                                                       |
| **`reg_alpha`**             | `0`                 | Regularização L1 aplicada aos pesos das folhas.                                                                       |
| **`scale_pos_weight`**      | `1`                 | Balanceia classes desbalanceadas (ex: fração negativa/positiva).                                                      |
| **`booster`**               | `'gbtree'`          | Tipo de modelo: `'gbtree'`, `'gblinear'` ou `'dart'`.                                                                 |
| **`n_jobs`**                | `None`              | Número de núcleos usados em paralelo (`-1` usa todos).                                                                |
| **`objective`**             | `'binary:logistic'` | Função objetivo (ex: `'reg:squarederror'`, `'multi:softprob'`).                                                       |
| **`eval_metric`**           | `None`              | Métrica usada para avaliação (ex: `'logloss'`, `'error'`, `'auc'`).                                                   |
| **`random_state`**          | `None`              | Define semente aleatória para reprodutibilidade.                                                                      |
| **`early_stopping_rounds`** | `None`              | Interrompe o treino se a métrica de validação não melhorar após certo número de rodadas.                              |
| **`tree_method`**           | `'auto'`            | Método de construção da árvore (`'auto'`, `'hist'`, `'gpu_hist'` para GPU).                                           |
| **`max_delta_step`**        | `0`                 | Passo máximo permitido na atualização dos pesos (usado em classificação desbalanceada).                               |

---

💡 **Em resumo:**

> O **XGBoost** é uma evolução do Gradient Boosting que adiciona **regularização**, **paralelismo** e **otimizações numéricas**.
> Ele cria árvores **sequenciais**, cada uma corrigindo o erro das anteriores, e se destaca por sua **precisão e eficiência computacional**.
> É o queridinho das **competições de dados** e um dos modelos mais utilizados na prática profissional.

#### **Treinando Algoritmo**

In [ ]:
from xgboost import XGBClassifier

In [ ]:
xgb = XGBClassifier(eval_metric='logloss', random_state=3, max_depth=4, n_estimators=200, learning_rate=0.05, objective='binary:logistic')
xgb.fit(x_treino, y_treino)

#### **Análise do Algoritmo**

##### **Análise de teste**

In [ ]:
previsoes_teste_xgb = xgb.predict(x_teste)
previsoes_teste_xgb

In [ ]:
y_teste.values

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
print(f'Acurácia do XGBoost (teste) é de: {accuracy_score(y_teste, previsoes_teste_xgb)*100:.2f}%')

In [ ]:
confusion_mtx_xgb_teste = confusion_matrix(y_teste, previsoes_teste_xgb)
confusion_mtx_xgb_teste

In [ ]:
# GRAFICO DA MATRIZ DE CONFUSÃO

z = confusion_mtx_xgb_teste

x = ['Predito Negativo', 'Predito Positivo']
y = ['Verdadeiro Negativo', 'Verdadeiro Positivo']

z_mask = [[1 if i == j else 0 for j in range(z.shape[1])] for i in range(z.shape[0])]
# Use the actual confusion numbers as annotation text
annot = z.astype(str).tolist()
# Colors: errors -> red, correct -> green
colorscale = [[0, "#d03d2a"], [1, "#73e526"]]

fig = ff.create_annotated_heatmap(
   z=z_mask,
   annotation_text=annot,
   
   x=x,
   y=y,
   colorscale=colorscale,
   showscale=False
)
# Make annotation text white for readability on colored cells
for a in fig.layout.annotations:
   a.font = dict(color='black', size=14)

fig.update_layout(title='Matriz de Confusão Teste - XGBoost', title_x=0.5 )


fig.show()

In [ ]:
print(classification_report(y_teste, previsoes_teste_xgb))

##### **Análise de treino**

In [ ]:
previsoes_treino_xgb = xgb.predict(x_treino)
previsoes_treino_xgb

In [ ]:
y_treino.values

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
print(f'Acurácia do XGBoost (treino) é de: {accuracy_score(y_treino, previsoes_treino_xgb)*100:.2f}%')

In [ ]:
matriz_confusao_xgb_treino = confusion_matrix(y_treino, previsoes_treino_xgb)
matriz_confusao_xgb_treino

In [ ]:
# GRAFICO DA MATRIZ DE CONFUSÃO

z = matriz_confusao_xgb_treino

x = ['Predito Negativo', 'Predito Positivo']
y = ['Verdadeiro Negativo', 'Verdadeiro Positivo']

z_mask = [[1 if i == j else 0 for j in range(z.shape[1])] for i in range(z.shape[0])]
# Use the actual confusion numbers as annotation text
annot = z.astype(str).tolist()
# Colors: errors -> red, correct -> green
colorscale = [[0, "#d03d2a"], [1, "#73e526"]]

fig = ff.create_annotated_heatmap(
   z=z_mask,
   annotation_text=annot,
   
   x=x,
   y=y,
   colorscale=colorscale,
   showscale=False
)
# Make annotation text white for readability on colored cells
for a in fig.layout.annotations:
   a.font = dict(color='black', size=14)

fig.update_layout(title='Matriz de Confusão Treino - XGBoost', title_x=0.5 )


fig.show()

In [ ]:
print(classification_report(y_treino, previsoes_treino_xgb))

#### **Validação Cruzada**

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

In [ ]:
kfold = KFold(n_splits=30, shuffle=True, random_state=0)
modelo_xgb = XGBClassifier(eval_metric='logloss', random_state=3, max_depth=4, n_estimators=200, learning_rate=0.05)
resultado_xgb = cross_val_score(modelo_xgb, previsores3_esc, target, cv=kfold)
acuracia_xgb = resultado_xgb.mean()
print(f'Accuracy médio XGBoost: {acuracia_xgb*100:.2f}%')

## **LightGBM**


[Documentação LightGBM](https://lightgbm.readthedocs.io/en/stable/)

---
🧠 **Mini-resumo LightGBM (Light Gradient Boosting Machine)**

O **LightGBM** é um algoritmo de **gradient boosting** desenvolvido pela **Microsoft**, otimizado para **grandes volumes de dados** e **alta velocidade de treinamento**.
Ele é uma alternativa moderna ao **XGBoost**, com foco em **eficiência**, **baixo uso de memória** e **suporte a GPU**.

---

⚙️ **Conceito Matemático**

Assim como o XGBoost, o LightGBM constrói árvores **sequencialmente**, onde cada nova árvore tenta **corrigir os erros residuais** das anteriores.

A predição é feita como:

$$
\hat{y}*i = \sum*{k=1}^{K} f_k(x_i), \quad f_k \in \mathcal{F}
$$

Mas a diferença está em **como as árvores são construídas**:

* O XGBoost faz **crescimento nível a nível (level-wise)**, expandindo todos os nós de um nível antes de ir para o próximo;
* O LightGBM usa **crescimento folha a folha (leaf-wise)**, sempre expandindo o **nó com maior ganho de informação**, o que gera **melhor precisão com menos árvores** (mas maior risco de overfitting).

---

⚡ **Diferenciais do LightGBM**

1. **Leaf-wise growth** → mais eficiente e preciso que o level-wise.
2. **Histogram-based algorithm** → converte valores contínuos em bins discretos, acelerando cálculos.
3. **Suporte nativo a GPU** → acelera o treinamento em bases grandes.
4. **Parallel e distributed training** → roda em várias CPUs ou clusters.

---

🎯 **Vantagens**

* Treinamento extremamente rápido;
* Escalável para milhões de linhas e centenas de features;
* Suporte a *categorical features* nativo;
* Alta acurácia mesmo com poucas árvores.

---

⚠️ **Limitações**

* Pode **overfitar** em bases pequenas se não regularizado;
* Leaf-wise growth pode gerar **árvores desbalanceadas**;
* Requer ajuste cuidadoso de hiperparâmetros para estabilidade.

---

⚙️ **Principais Hiperparâmetros do **`LGBMClassifier`****

| Parâmetro               | Valor Padrão       | Descrição                                                                                            |
| ----------------------- | ------------------ | ---------------------------------------------------------------------------------------------------- |
| **`num_leaves`**        | `31`               | Número máximo de folhas por árvore. Aumentar melhora a precisão, mas aumenta o risco de overfitting. |
| **`max_depth`**         | `-1`               | Profundidade máxima da árvore. `-1` significa ilimitada.                                             |
| **`learning_rate`**     | `0.1`              | Taxa de aprendizado. Controla o peso de cada nova árvore.                                            |
| **`n_estimators`**      | `100`              | Número de árvores (iterações de boosting).                                                           |
| **`min_data_in_leaf`**  | `20`               | Mínimo de amostras em cada folha. Valores maiores reduzem overfitting.                               |
| **`feature_fraction`**  | `1.0`              | Fração de features usadas em cada árvore. Menores valores reduzem overfitting.                       |
| **`bagging_fraction`**  | `1.0`              | Fração de amostras usadas em cada iteração (como `subsample` no XGBoost).                            |
| **`bagging_freq`**      | `0`                | Frequência (em iterações) para aplicar bagging. 0 = desativado.                                      |
| **`lambda_l1`**         | `0.0`              | Regularização L1 (Lasso) nos pesos das folhas.                                                       |
| **`lambda_l2`**         | `0.0`              | Regularização L2 (Ridge) nos pesos das folhas.                                                       |
| **`min_split_gain`**    | `0.0`              | Ganho mínimo exigido para dividir um nó (regularizador).                                             |
| **`boosting_type`**     | `'gbdt'`           | Tipo de boosting (`'gbdt'`, `'dart'`, `'goss'`).                                                     |
| **`objective`**         | `'binary'`         | Função objetivo: `'binary'`, `'multiclass'`, `'regression'`, etc.                                    |
| **`metric`**            | `'binary_logloss'` | Métrica de avaliação.                                                                                |
| **`colsample_bytree`**  | `1.0`              | Sinônimo de `feature_fraction`.                                                                      |
| **`subsample`**         | `1.0`              | Sinônimo de `bagging_fraction`.                                                                      |
| **`subsample_freq`**    | `0`                | Sinônimo de `bagging_freq`.                                                                          |
| **`max_bin`**           | `255`              | Número máximo de bins usados para discretização.                                                     |
| **`min_gain_to_split`** | `0.0`              | Ganho mínimo necessário para dividir um nó.                                                          |
| **`n_jobs`**            | `-1`               | Número de threads usadas em paralelo.                                                                |
| **`verbosity`**         | `-1`               | Nível de log (-1 = silencioso).                                                                      |
| **`random_state`**      | `None`             | Semente aleatória para reprodutibilidade.                                                            |

---

💡 **Em resumo:**

> O **LightGBM** é uma versão otimizada do Gradient Boosting que usa **crescimento folha a folha**, **histogramas discretos** e **processamento paralelo**.
> Ele é **muito mais rápido** que o XGBoost, mantendo **alta precisão** e **eficiência em grandes volumes de dados**.
> Ideal para aplicações em que **tempo e escala** são críticos — sem abrir mão de performance.

#### **Treinando Algoritmo**

In [ ]:
import lightgbm as lgb

In [ ]:
# Hiperparâmetros devem ser lançandos em um dicionário
hiperparametros = {
   'objective': 'binary',
   'metric': 'binary_logloss',
   'learning_rate': 0.05,
   'num_leaves': 150,
   'max_depth': 7,
   'min_data_in_leaf': 10,
   'min_gain_to_split': 0,
   'feature_pre_filter': False,
   'verbose': -1
}

In [ ]:
# Dataset de Treino
dataset = lgb.Dataset(x_treino, label=y_treino)

In [ ]:
lgbm = lgb.train(hiperparametros, dataset, num_boost_round=200)

#### **Análise do Algoritmo**

##### **Análise de Teste**

In [ ]:
# Marcação do tempo de execução
from datetime import datetime
inicio = datetime.now()
lgbm = lgb.train(hiperparametros, dataset, num_boost_round=200)
fim = datetime.now()
tempo_execucao = fim - inicio
tempo_execucao

In [ ]:
# Previsores teste LGBM
previsoes_teste_lgbm = lgbm.predict(x_teste)
previsoes_teste_lgbm

In [ ]:
previsoes_teste_lgbm.shape

In [ ]:
# Quando for menor que 0.5, classifica como 0, caso contrário, classifica como 1
for i in range(len(previsoes_teste_lgbm)):
   if previsoes_teste_lgbm[i] < 0.5:
      previsoes_teste_lgbm[i] = 0
   else:
      previsoes_teste_lgbm[i] = 1
np.array(previsoes_teste_lgbm)

In [ ]:
y_teste.values

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
print(f'Acurácia do LGBM (teste) é de: {accuracy_score(y_teste, previsoes_teste_lgbm)*100:.2f}%')

In [ ]:
matriz_confusao_lgbm_teste = confusion_matrix(y_teste, previsoes_teste_lgbm)
matriz_confusao_lgbm_teste

In [ ]:
# GRAFICO DA MATRIZ DE CONFUSÃO

z = matriz_confusao_lgbm_teste

x = ['Predito Negativo', 'Predito Positivo']
y = ['Verdadeiro Negativo', 'Verdadeiro Positivo']

z_mask = [[1 if i == j else 0 for j in range(z.shape[1])] for i in range(z.shape[0])]
# Use the actual confusion numbers as annotation text
annot = z.astype(str).tolist()
# Colors: errors -> red, correct -> green
colorscale = [[0, "#d03d2a"], [1, "#73e526"]]

fig = ff.create_annotated_heatmap(
   z=z_mask,
   annotation_text=annot,
   
   x=x,
   y=y,
   colorscale=colorscale,
   showscale=False
)
# Make annotation text white for readability on colored cells
for a in fig.layout.annotations:
   a.font = dict(color='black', size=14)

fig.update_layout(title='Matriz de Confusão Teste - LightGBM', title_x=0.5 )


fig.show()

In [ ]:
print(classification_report(y_teste, previsoes_teste_lgbm))

##### **Análise de Treino**

In [ ]:
# Previsores treino LGBM
previsoes_treino_lgbm = lgbm.predict(x_treino)
previsoes_treino_lgbm

In [ ]:
# Quando for menor que 0.5, classifica como 0, caso contrário, classifica como 1
for i in range(len(previsoes_treino_lgbm)):
   if previsoes_treino_lgbm[i] < 0.5:
      previsoes_treino_lgbm[i] = 0
   else:
      previsoes_treino_lgbm[i] = 1
np.array(previsoes_treino_lgbm)

In [ ]:
y_treino.values

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
print(f'Acurácia do LGBM (treino) é de: {accuracy_score(y_treino, previsoes_treino_lgbm)*100:.2f}% ')

In [ ]:
matriz_confusao_lgbm_treino = confusion_matrix(y_treino, previsoes_treino_lgbm)
matriz_confusao_lgbm_treino

In [ ]:
# GRAFICO DA MATRIZ DE CONFUSÃO

z = matriz_confusao_lgbm_treino

x = ['Predito Negativo', 'Predito Positivo']
y = ['Verdadeiro Negativo', 'Verdadeiro Positivo']

z_mask = [[1 if i == j else 0 for j in range(z.shape[1])] for i in range(z.shape[0])]
# Use the actual confusion numbers as annotation text
annot = z.astype(str).tolist()
# Colors: errors -> red, correct -> green
colorscale = [[0, "#d03d2a"], [1, "#73e526"]]

fig = ff.create_annotated_heatmap(
   z=z_mask,
   annotation_text=annot,
   
   x=x,
   y=y,
   colorscale=colorscale,
   showscale=False
)
# Make annotation text white for readability on colored cells
for a in fig.layout.annotations:
   a.font = dict(color='black', size=14)

fig.update_layout(title='Matriz de Confusão Treino - LightGBM', title_x=0.5 )


fig.show()

In [ ]:
print(classification_report(y_treino, previsoes_treino_lgbm))

#### **Verificação Cruzada**

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

In [ ]:
kfold = KFold(n_splits=30, shuffle=True, random_state=0)
modelo_lgbm = lgb.LGBMClassifier(objective= 'binary',
   metric= 'binary_logloss',
   learning_rate= 0.05,
   num_leaves= 150,
   max_depth= 7,
   min_data_in_leaf= 10,
   min_gain_to_split= 0,
   feature_pre_filter= False,
   verbose= -1)
resultado_lgbm = cross_val_score(modelo_lgbm, previsores3_esc, target, cv=kfold)
acuraria_lgbm = resultado_lgbm.mean()
print(f'Accuracy médio LGBM: {acuraria_lgbm*100:.2f}%')

## **CatBoost (Categorical Boosting)**

[Documentação CatBoost](https://catboost.ai/docs/en/)

---

🧠 **Mini-resumo CatBoost (Categorical Boosting)**

O **CatBoost** é um algoritmo de **gradient boosting** desenvolvido pela **Yandex**, projetado especialmente para lidar com **features categóricas** de forma nativa e automática.
O nome vem de *“Categorical Boosting”*, e ele combina **alta precisão**, **baixo pré-processamento** e **treinamento rápido e estável**. Usado em aprendizado supervisionado, trabalhando tanto com classificação quanto por regressão.

---

⚙️ **Conceito Matemático**

Assim como o XGBoost e o LightGBM, o CatBoost treina **múltiplas árvores sequenciais**, onde cada árvore corrige os erros das anteriores.

A predição final é dada por:

$$
\hat{y}*i = \sum*{t=1}^{T} f_t(x_i)
$$

onde 
$$
f_t(x)
$$
representa a ( t )-ésima árvore e ( T ) é o número total de iterações (árvores).

O objetivo é **minimizar uma função de perda** com regularização, geralmente via *gradient boosting*:

$$
L = \sum_{i=1}^{n} l(y_i, \hat{y}*i) + \lambda \sum*{t=1}^{T} ||f_t||^2
$$

---

🧩 **O diferencial do CatBoost**

1. **Trata variáveis categóricas automaticamente** – converte-as internamente em combinações de estatísticas (*target encoding* controlado).
2. **Evita overfitting por permutação aleatória** – usa *Ordered Boosting*, uma técnica que evita vazamento de informação (data leakage).
3. **Dispensa one-hot encoding** – o que economiza tempo e memória.
4. **Treina rápido e com menos ajustes** – os hiperparâmetros padrão já funcionam bem em muitos cenários.

---

🎯 **Vantagens**

* Suporte nativo a variáveis categóricas;
* Menor necessidade de *tuning*;
* Estável e robusto a overfitting;
* Compatível com CPU e GPU;
* Funciona bem com datasets pequenos ou médios;
* Pode ser usado em problemas de classificação e regressão;
* Pode lidar com valores ausentes;
* Processa automaticamente as variáveis categóricas.

---

⚠️ **Limitações**

* Treinamento mais lento que o LightGBM em bases enormes;
* Pode consumir mais memória;
* Pouca transparência nos detalhes internos das transformações categóricas.

---

⚙️ **Principais Hiperparâmetros do **`CatBoostClassifier`****

| Parâmetro                   | Valor Padrão      | Descrição                                                                 |
| --------------------------- | ----------------- | ------------------------------------------------------------------------- |
| **`iterations`**            | `1000`            | Número total de árvores (iterações de boosting).                          |
| **`learning_rate`**         | `0.03`            | Taxa de aprendizado (quanto cada nova árvore contribui).                  |
| **`depth`**                 | `6`               | Profundidade máxima das árvores.                                          |
| **`l2_leaf_reg`**           | `3.0`             | Regularização L2 aplicada aos pesos das folhas.                           |
| **`loss_function`**         | `'Logloss'`       | Função de perda (ex: `'Logloss'`, `'CrossEntropy'`, `'RMSE'`, `'MAE'`).   |
| **`eval_metric`**           | `'Logloss'`       | Métrica usada na validação.                                               |
| **`bootstrap_type`**        | `'Bayesian'`      | Tipo de amostragem: `'Bayesian'`, `'Bernoulli'`, `'Poisson'` etc.         |
| **`subsample`**             | `0.66`            | Fração das amostras usadas em cada iteração.                              |
| **`rsm`**                   | `1.0`             | Fração das features usadas em cada árvore. (Random Subspace Method)       |
| **`random_strength`**       | `1.0`             | Intensidade da aleatoriedade ao dividir nós (controla overfitting).       |
| **`bagging_temperature`**   | `1.0`             | Controla o grau de amostragem bayesiana.                                  |
| **`border_count`**          | `254`             | Número de divisões (bins) usados para transformar variáveis contínuas.    |
| **`grow_policy`**           | `'SymmetricTree'` | Estratégia de crescimento da árvore (`'SymmetricTree'` ou `'Depthwise'`). |
| **`verbose`**               | `True`            | Exibe o progresso do treinamento.                                         |
| **`task_type`**             | `'CPU'`           | Define o uso de CPU ou GPU. (`'GPU'` acelera muito o treino).             |
| **`cat_features`**          | `None`            | Lista de colunas categóricas (pode ser detectado automaticamente).        |
| **`random_seed`**           | `None`            | Define semente aleatória para reprodutibilidade.                          |
| **`early_stopping_rounds`** | `None`            | Interrompe o treino se não houver melhora na métrica.                     |
| **`custom_metric`**         | `None`            | Permite métricas personalizadas para monitoramento.                       |

---

💡 **Em resumo:**

> O **CatBoost** é o algoritmo de boosting mais **inteligente com variáveis categóricas**.
> Ele combina **simplicidade de uso**, **robustez**, e **excelente desempenho** com menos necessidade de pré-processamento e *tuning*.
> Ideal quando há **muitas features categóricas**, **datasets médios**, e necessidade de **precisão estável** sem muito esforço.

#### **Pré-processando**

In [ ]:
df

In [ ]:
previsores_catboost = df.iloc[:,0:11]
previsores_catboost

In [ ]:
target_catboost = df.iloc[:,11]
target_catboost

#### **Base de Treino CatBoost**

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x_treino, x_teste, y_treino, y_teste = train_test_split(previsores_catboost, target_catboost, test_size=0.3, random_state=0)

In [ ]:
categoricas = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']

#### **Treinando Algoritmo**

In [ ]:
from catboost import CatBoostClassifier

In [ ]:
catboost = CatBoostClassifier(task_type='CPU', iterations=80, learning_rate=0.1, depth=8, random_state=5, eval_metric='Accuracy')

In [ ]:
catboost.fit(x_treino,y_treino, cat_features=categoricas, plot=True, eval_set=(x_teste, y_teste))

#### **Análise do Algoritmo**

##### **Análise de Teste**

In [ ]:
previsao_cat_teste = catboost.predict(x_teste)
previsao_cat_teste

In [ ]:
y_teste.values

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
print(f'Acurácia do CatBoost (teste) é de: {accuracy_score(y_teste, previsao_cat_teste)*100:.2f}%')

In [ ]:
matriz_confusao_cat_teste = confusion_matrix(y_teste, previsao_cat_teste)
matriz_confusao_cat_teste

In [ ]:
# GRAFICO DA MATRIZ DE CONFUSÃO

z = matriz_confusao_cat_teste

x = ['Predito Negativo', 'Predito Positivo']
y = ['Verdadeiro Negativo', 'Verdadeiro Positivo']

z_mask = [[1 if i == j else 0 for j in range(z.shape[1])] for i in range(z.shape[0])]
# Use the actual confusion numbers as annotation text
annot = z.astype(str).tolist()
# Colors: errors -> red, correct -> green
colorscale = [[0, "#d03d2a"], [1, "#73e526"]]

fig = ff.create_annotated_heatmap(
   z=z_mask,
   annotation_text=annot,
   
   x=x,
   y=y,
   colorscale=colorscale,
   showscale=False
)
# Make annotation text white for readability on colored cells
for a in fig.layout.annotations:
   a.font = dict(color='black', size=14)

fig.update_layout(title='Matriz de Confusão Teste - CatBoost', title_x=0.5 )


fig.show()

In [ ]:
print(classification_report(y_teste, previsao_cat_teste))

##### **Análise Treino**

In [ ]:
previsao_cat_treino = catboost.predict(x_treino)
previsao_cat_treino

In [ ]:
y_treino.values

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
print(f'Acurácia do CatBoost (treino) é de: {accuracy_score(y_treino, previsao_cat_treino)*100:.2f}%')

In [ ]:
matriz_confusao_cat_treino = confusion_matrix(y_treino, previsao_cat_treino)
matriz_confusao_cat_treino

In [ ]:
# GRAFICO DA MATRIZ DE CONFUSÃO

z = matriz_confusao_cat_treino

x = ['Predito Negativo', 'Predito Positivo']
y = ['Verdadeiro Negativo', 'Verdadeiro Positivo']

z_mask = [[1 if i == j else 0 for j in range(z.shape[1])] for i in range(z.shape[0])]
# Use the actual confusion numbers as annotation text
annot = z.astype(str).tolist()
# Colors: errors -> red, correct -> green
colorscale = [[0, "#d03d2a"], [1, "#73e526"]]

fig = ff.create_annotated_heatmap(
   z=z_mask,
   annotation_text=annot,
   
   x=x,
   y=y,
   colorscale=colorscale,
   showscale=False
)
# Make annotation text white for readability on colored cells
for a in fig.layout.annotations:
   a.font = dict(color='black', size=14)

fig.update_layout(title='Matriz de Confusão Treino - CatBoost', title_x=0.5 )


fig.show()

In [ ]:
print(classification_report(y_treino, previsao_cat_treino))

#### **Validação Cruzada**

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

In [ ]:
acuracia_treino_teste_cat = [accuracy_score(y_teste, previsao_cat_teste), accuracy_score(y_treino, previsao_cat_treino)]

In [ ]:
kfold = KFold(n_splits=30, shuffle=True, random_state=0)
modelo_cat = CatBoostClassifier(task_type='CPU', iterations=80, learning_rate=0.1, depth=8, random_state=5, eval_metric='Accuracy')
resultado_cat = cross_val_score(modelo_cat, previsores, target, cv=kfold)
acuracia_cat = min(acuracia_treino_teste_cat)
acuracia_media_cat = resultado_cat.mean()
print(f'CatBoost:\nAcurácia (treino/ teste) {acuracia_cat*100:.2f}%\nAcurácia média CatBoost: {acuracia_media_cat*100:.2f}%')

In [ ]:
# Gráfico Acurácia Treino e Teste
modelos = ['Naive Bayes', 'SVC', 'Logística', 'KNN', 'Árvore Decisão', 'Random Forest', 'XGBoost', 'LightGBM', 'CatBoost']
acuracias_treino_teste = [acuracia_naive*100, acuracia_svc*100, acuracia_logistica*100, acuracia_knn*100, acuracia_tree*100, acuracia_random*100, acuracia_xgb*100, acuraria_lgbm*100, acuracia_cat*100]
x = np.arange(len(modelos))
largura = 0.35
fig, ax = plt.subplots(figsize=(15,6))
barras1 = ax.bar(x - largura/2, acuracias_treino_teste, largura, label='Acurácia (Treino e Teste)', color='b')
ax.set_ylabel('Acurácia (%)', fontsize=14)
ax.set_title('Acurácia dos Modelos - Treino e Teste', fontsize=16)
ax.set_xticks(x)  
ax.set_xticklabels(modelos)
ax.set_ylim(0, 100)
for barra in barras1:
   altura = barra.get_height()
   ax.annotate(f'{altura:.2f}%',
               xy=(barra.get_x() + barra.get_width() / 2, altura),
               xytext=(0, 3),  
               textcoords="offset points",
               ha='center', va='bottom', fontsize=12)
plt.show()

In [ ]:
# GRAFICO ACURÁCIAS DOS MODELOS VALIDAÇÃO CRUZADA
modelos = ['Naive Bayes', 'SVC', 'Logística', 'KNN', 'Árvore Decisão', 'Random Forest', 'XGBoost', 'LightGBM', 'CatBoost']
acuracias = [acuracia_naive*100, acuracia_svc*100, acuracia_logistica*100, acuracia_knn*100, acuracia_tree*100, acuracia_random*100, acuracia_xgb*100, acuraria_lgbm*100, acuracia_media_cat*100]
plt.figure(figsize=(15,6))
sns.barplot(x=modelos, y=acuracias, palette='viridis')
plt.ylim(0, 100)
plt.title('Acurácia dos Modelos - Validação Cruzada', fontsize=16)
plt.ylabel('Acurácia (%)', fontsize=14)
plt.xlabel('Modelos', fontsize=14)
for i in range(len(modelos)):
   plt.text(i, acuracias[i] + 0.5, f'{acuracias[i]:.2f}%', ha='center', fontsize=12)
plt.show()

In [ ]:
# GRAFICO ACERTOS DOS MODDELOS, OU SEJA, SOMA DOS VERDADEIROS POSITIVOS E VERDADEIROS NEGATIVOS
modelos = ['Naive Bayes', 'SVC', 'Logística', 'KNN', 'Árvore Decisão', 'Random Forest', 'XGBoost', 'LightGBM', 'CatBoost']
acertos = [
   matriz_confusao_teste_nb[0,0] + matriz_confusao_teste_nb[1,1],
   confusion_mtx_svc_teste[0,0] + confusion_mtx_svc_teste[1,1],
   confusion_mtx_logica_teste[0,0] + confusion_mtx_logica_teste[1,1],
   confusion_matrix_knn_teste[0,0] + confusion_matrix_knn_teste[1,1],
   confusion_mtx_tree_teste[0,0] + confusion_mtx_tree_teste[1,1],
   confusion_matrix_randon_teste[0,0] + confusion_matrix_randon_teste[1,1],
   confusion_mtx_xgb_teste[0,0] + confusion_mtx_xgb_teste[1,1],
   matriz_confusao_lgbm_teste[0,0] + matriz_confusao_lgbm_teste[1,1],
   matriz_confusao_cat_teste[0,0] + matriz_confusao_cat_teste[1,1]
]
plt.figure(figsize=(15,6))
sns.barplot(x=modelos, y=acertos, palette='viridis')
plt.title('Número de Acertos dos Modelos (Teste)', fontsize=16)
plt.ylabel('Número de Acertos', fontsize=14)
plt.xlabel('Modelos', fontsize=14)
for i in range(len(modelos)):
   plt.text(i, acertos[i] + 5, f'{acertos[i]}', ha='center', fontsize=12)
plt.show()

## **Relatório Final de Desempenho dos Modelos**

A análise comparou nove algoritmos de classificação aplicados ao problema de predição de doença cardíaca. A avaliação considerou três métricas principais: acurácia no conjunto de treino e teste, acurácia por validação cruzada e número absoluto de acertos no conjunto de teste. A ideia foi observar não apenas o desempenho bruto, mas também a consistência entre diferentes formas de validação.

---

**Tabela Comparativa dos Resultados**

| Modelo              | Acurácia Treino/Teste (%) | Acurácia Validação Cruzada (%) | Acertos no Teste |
|--------------------|----------------------------|----------------------------------|-------------------|
| Naive Bayes        | 84.78                     | 84.78                           | 234               |
| SVC                | 85.61                     | 85.61                           | 238               |
| Regressão Logística| 85.83                     | 85.83                           | 238               |
| KNN                | 85.83                     | 85.83                           | 234               |
| Árvore de Decisão  | 81.13                     | 81.13                           | 223               |
| Random Forest      | 86.06                     | 86.06                           | 234               |
| XGBoost            | 87.04                     | 87.04                           | 237               |
| LightGBM           | 86.84                     | 86.84                           | 234               |
| **CatBoost**        | **86.96**                 | **87.70**                       | **240**           |

---

**Interpretação Geral**

Os resultados mostram uma relação harmoniosa entre as métricas. Modelos que tiveram bom desempenho no conjunto de treino e teste também apresentaram números muito próximos na validação cruzada, o que reforça estabilidade e reduz a chance de overfitting. A principal exceção foi a Árvore de Decisão, que ficou claramente abaixo dos demais, algo esperado pela sensibilidade desse algoritmo a variações nos dados.

O trio de melhor performance foi formado por CatBoost, XGBoost e LightGBM. Esses modelos mostraram alta eficiência no tratamento de relações não lineares e variáveis categóricas, o que se refletiu no desempenho final.

---

**Destaque para o CatBoost**

O CatBoost se destacou de forma clara.

Ele apresentou:

- **Acurácia de 86,96 por cento no conjunto de teste**
- **Acurácia de 87,70 por cento na validação cruzada**
- **Maior número absoluto de acertos: 240**

Esse padrão indica que o modelo generaliza bem e não depende exclusivamente de padrões específicos do conjunto de treino. Seu tratamento interno de variáveis categóricas também contribui para maior estabilidade.

---

**Desempenho Intermediário**

SVC, Regressão Logística e KNN mantiveram resultados sólidos, com acurácia entre 85 e 86 por cento. Eles formam um grupo equilibrado e previsível que entrega bons resultados sem depender de algoritmos complexos. Esse comportamento sugere que o dataset é relativamente bem distribuído e que relações lineares ainda capturam boa parte da estrutura dos dados.

O Naive Bayes veio logo abaixo, com desempenho competitivo, embora limitado pela suposição de independência entre variáveis que raramente ocorre em cenários reais.

---
**Modelo de Menor Desempenho**

A Árvore de Decisão obteve os menores números nas três métricas avaliadas. Ainda assim, ela é útil para interpretação, já que permite visualizar regras explícitas de decisão, embora não tenha se mostrado a melhor opção para performance neste caso.

---

**Conclusão**

Os testes demonstram que o **CatBoost** oferece o melhor equilíbrio entre precisão, consistência e número de acertos. Ele é seguido de perto por XGBoost e LightGBM, que também entregam desempenho competitivo. Modelos clássicos como SVC e Regressão Logística permanecem opções confiáveis, enquanto a Árvore de Decisão funciona melhor como ferramenta interpretativa do que como candidata ao melhor desempenho.

O estudo mostra que a escolha do modelo ideal depende do objetivo final. Se a prioridade for maximizar acurácia com estabilidade, o CatBoost se torna a escolha mais adequada para esse conjunto de dados.
